# Semantic Joins — Final Experiments

**Project 4 · BasketballTeam / Country / Device**  
Traditional Matching (RapidFuzz) + Embedding Retrieval + Qwen Verification

> **這是正式整理版。** 原本的 `Big_data (2).ipynb` 包含 BasketballTeam 的開發與除錯；
> Country / Device 來自另一份實驗 Notebook。本檔案將三個 Dataset 的**最終評估與重現方法**合併成一份，
> 不包含重複的測試、錯誤的安裝指令或重複載入模型。

### 執行方式

1. 在 Google Colab 開啟本 Notebook。連結你原本的 Google Drive。
2. **依序執行第 0–5 節**：載入既有 CSV、重新計算指標、輸出分析。**不會重跑 Qwen 或 Embedding。**
3. 第 6 節是**可選的演算法實作**，方便老師檢查方法；只有主動呼叫那些函式才會載入模型。

### 固定實驗設定

| Component | Setting |
|---|---|
| Fuzzy Matching | RapidFuzz `fuzz.ratio`，小寫 + 去除首尾空白，threshold **80** |
| Candidate Retrieval | `sentence-transformers/all-MiniLM-L6-v2`，normalize embeddings，Right → Left Top-**5** |
| LLM | `Qwen/Qwen2.5-1.5B-Instruct`；同一份 generic identity prompt |
| Outputs | `YES` / `NO` / `UNSURE`；其餘視為 `ABSTAIN`（不當成 match） |
| Data source | [AutomaticFuzzyJoin benchmark](https://github.com/chu-data-lab/AutomaticFuzzyJoin) |

### 已完成的正式結果（即使尚未執行 Cell，也可以直接在 GitHub 閱讀）

| Dataset | Traditional F1 | Qwen-only F1 | Fixed Hybrid F1 | Full Hybrid F1 | Fixed Hybrid LLM 節省 |
|---|---:|---:|---:|---:|---:|
| BasketballTeam | 0.4858 | 0.5633 | 0.5910 | 0.5924 | 9.04% |
| Country | 0.2446 | 0.4242 | 0.4152 | 0.3028 | 14.09% |
| Device | 0.2955 | 0.3604 | 0.3575 | 0.2775 | 26.60% |

Recorded Qwen inference time (Qwen-only → Fixed Hybrid): **71.93 → 65.26 s** (BasketballTeam),
**127.26 → 107.36 s** (Country), **282.50 → 209.34 s** (Device).
These are **per-pair inference sums**, not end-to-end runtimes.

BasketballTeam 用於初步 threshold 探索；Country 和 Device 固定採用 threshold 80、Top-5，不根據測試集調參。

## 0. 共用設定（只設定一次）

CSV 預設都在原本的 `/content/drive/MyDrive/big_data_project/`。**不會覆蓋原始 Qwen 預測 CSV**。
如果缺少某個檔案，請先從原本的 Drive 資料夾確認檔名。

新 Colab Runtime 如缺少 `rapidfuzz`，先執行下一格的輕量安裝即可。**不會更動 Transformers / Qwen。**

In [ ]:
# First-time Colab setup: this does NOT install or upgrade Transformers.
%pip install -q rapidfuzz

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
from rapidfuzz import fuzz
from IPython.display import display

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

SAVE_DIR = Path(os.environ.get(
    "SEMJOIN_RESULTS_DIR", "/content/drive/MyDrive/big_data_project"
))

REQUIRED_FILES = [
    "qwen_generic_top5.csv",
    "country_fuzzy80.csv", "country_top5_candidates.csv",
    "country_llm_queue.csv", "country_qwen_top5_full1455.csv",
    "device_fuzzy80.csv", "device_top5_candidates.csv",
    "device_llm_queue.csv", "device_qwen_top5_full3290.csv",
]
missing = [name for name in REQUIRED_FILES if not (SAVE_DIR / name).exists()]
if missing:
    raise FileNotFoundError(
        f"找不到這些 CSV（路徑：{SAVE_DIR}）：\n" + "\n".join(missing)
    )
print("✅ 已找到實驗 CSV。資料夾：", SAVE_DIR)

In [ ]:
# 共用函式：避免每個 Dataset 重複貼上同一套計算
BENCHMARK_URL = (
    "https://raw.githubusercontent.com/chu-data-lab/AutomaticFuzzyJoin/"
    "master/src/autofj/benchmark/"
)

def load_benchmark(dataset):
    base = BENCHMARK_URL + dataset + "/"
    left = pd.read_csv(base + "left.csv")
    right = pd.read_csv(base + "right.csv")
    ground_truth = pd.read_csv(base + "gt.csv")
    truth = set(zip(ground_truth["id_l"].astype(int),
                    ground_truth["id_r"].astype(int)))
    return left, right, truth

def read_pairs(filename):
    df = pd.read_csv(SAVE_DIR / filename)
    return set(zip(df["left_id"].astype(int), df["right_id"].astype(int)))

def pairs_from(df):
    return set(zip(df["left_id"].astype(int), df["right_id"].astype(int)))

def read_predictions(filename, expected_pairs=None):
    df = pd.read_csv(SAVE_DIR / filename, keep_default_na=False)
    assert not df.duplicated(["left_id", "right_id"]).any(), filename
    if expected_pairs is not None:
        assert pairs_from(df) == expected_pairs, f"候選集合與 {filename} 不一致"
    valid = ["YES", "NO", "UNSURE"]
    label = df["prediction"].astype(str).str.strip().str.upper()
    df["prediction_eval"] = label.where(label.isin(valid), "ABSTAIN")
    return df

def accepted(df):
    return pairs_from(df.loc[df["prediction_eval"].eq("YES")])

def compute_fuzzy(left, right, threshold=80):
    # 和原始實驗相同：所有 Left × Right pair 都比一次
    right_items = [(int(x.id), str(x.title).lower().strip())
                   for x in right.itertuples(index=False)]
    result = set()
    for x in left.itertuples(index=False):
        la = str(x.title).lower().strip()
        for r_id, rb in right_items:
            if fuzz.ratio(la, rb) >= threshold:
                result.add((int(x.id), r_id))
    return result

def evaluate(dataset, method, predicted, truth, llm_calls):
    tp, fp, fn = (len(predicted & truth),
                  len(predicted - truth), len(truth - predicted))
    p = tp / (tp + fp) if tp + fp else 0
    r = tp / (tp + fn) if tp + fn else 0
    f1 = 2 * p * r / (p + r) if p + r else 0
    return {
        "Dataset": dataset, "Method": method,
        "TP": tp, "FP": fp, "FN": fn, "LLM Calls": llm_calls,
        "Precision": round(p, 4), "Recall": round(r, 4),
        "F1": round(f1, 4)
    }

result_rows = []
print("✅ Shared helpers ready")

## 1. BasketballTeam — 開發資料集

原本 `Big_data (2).ipynb` 主要就是這一段。已合併重複的 RapidFuzz/Qwen Cell。

- 資料集：Left 928、Right 166、Ground Truth 166
- Threshold 探索：70 / **80** / 90
- Embedding Top-5：**830** 候選、覆蓋 **139** 組正確配對
- 原始 `qwen_generic_top5.csv` 是 830 組；另外補驗 6 組的檔案 `qwen_full836.csv` 如存在，會額外顯示比較

In [ ]:
basket_left, basket_right, basket_truth = load_benchmark("BasketballTeam")

threshold_rows = []
for threshold in [70, 80, 90]:
    current = compute_fuzzy(basket_left, basket_right, threshold)
    row = evaluate("BasketballTeam", f"Traditional {threshold}",
                   current, basket_truth, 0)
    threshold_rows.append(row)
    if threshold == 80:
        basket_fuzzy80 = current

print("BasketballTeam — RapidFuzz threshold analysis")
display(pd.DataFrame(threshold_rows)[
    ["Method", "TP", "FP", "FN", "Precision", "Recall", "F1"]
])

In [ ]:
basket_qwen = read_predictions("qwen_generic_top5.csv")
basket_top5 = pairs_from(basket_qwen)
assert len(basket_top5) == 830

basket_queue = basket_top5 - basket_fuzzy80
basket_yes = accepted(basket_qwen)
basket_fixed = (basket_fuzzy80 & basket_top5) | (basket_yes & basket_queue)
basket_full = basket_fuzzy80 | (basket_yes & basket_queue)

result_rows += [
    evaluate("BasketballTeam", "Traditional 80", basket_fuzzy80, basket_truth, 0),
    evaluate("BasketballTeam", "Qwen-only Top-5", basket_yes, basket_truth, 830),
    evaluate("BasketballTeam", "Fixed Hybrid 80", basket_fixed, basket_truth,
             len(basket_queue)),
    evaluate("BasketballTeam", "Full Hybrid 80", basket_full, basket_truth,
             len(basket_queue)),
]

print("BasketballTeam — final comparison")
print("Top-5 correct pairs:", len(basket_top5 & basket_truth))
print("Fuzzy outside Top-5:", len(basket_fuzzy80 - basket_top5))
display(pd.DataFrame(result_rows[-4:]).drop(columns="Dataset"))

# Full Hybrid 加入 Top-5 外的 6 組 Fuzzy 配對，候選集合比 Qwen-only 大。
# 可選：如存在已經跑完的 836 Qwen 結果，顯示公平的 expanded-candidate baseline。
expanded_file = SAVE_DIR / "qwen_full836.csv"
if expanded_file.exists():
    basket_expanded = read_predictions("qwen_full836.csv")
    expected_expanded = basket_top5 | (basket_fuzzy80 - basket_top5)
    assert pairs_from(basket_expanded) == expected_expanded
    print("Expanded-candidate Qwen-only baseline (836 pairs):")
    display(pd.DataFrame([evaluate(
        "BasketballTeam", "Qwen-only expanded 836",
        accepted(basket_expanded), basket_truth, len(basket_expanded)
    )]))

## 2. Country — 固定參數的泛化測試

- Left 2,791；Right 291；Ground Truth 291
- 取用已保存的 `country_fuzzy80.csv`、`country_top5_candidates.csv`、`country_llm_queue.csv`
- **Qwen-only** 使用完整 1,455 筆 Top-5 預測；**Fixed Hybrid** 只需驗證其中 1,250 筆
- 原始的 4 筆無效 Qwen 回覆，在評估時視為 `ABSTAIN`

In [ ]:
country_left, country_right, country_truth = load_benchmark("Country")
country_fuzzy80 = read_pairs("country_fuzzy80.csv")
country_top5 = read_pairs("country_top5_candidates.csv")
country_queue = read_pairs("country_llm_queue.csv")
country_qwen = read_predictions("country_qwen_top5_full1455.csv", country_top5)
assert len(country_top5) == 1455
assert country_queue == country_top5 - country_fuzzy80

country_yes = accepted(country_qwen)
country_fixed = (country_fuzzy80 & country_top5) | (country_yes & country_queue)
country_full = country_fuzzy80 | (country_yes & country_queue)

result_rows += [
    evaluate("Country", "Traditional 80", country_fuzzy80, country_truth, 0),
    evaluate("Country", "Qwen-only Top-5", country_yes, country_truth, len(country_top5)),
    evaluate("Country", "Fixed Hybrid 80", country_fixed, country_truth,
             len(country_queue)),
    evaluate("Country", "Full Hybrid 80", country_full, country_truth,
             len(country_queue)),
]
print("Country — top-5 coverage:", len(country_top5 & country_truth), "/", len(country_truth))
print("Fuzzy outside top-5:", len(country_fuzzy80 - country_top5))
print("Invalid Qwen outputs:", (country_qwen["prediction_eval"] == "ABSTAIN").sum())
display(pd.DataFrame(result_rows[-4:]).drop(columns="Dataset"))

## 3. Device — 固定參數的泛化測試

- Left 6,933；Right 658；Ground Truth 658
- Embedding Top-5：3,290 組候選，覆蓋 545 組正確配對
- Qwen-only 驗證 3,290 組；Fixed Hybrid 驗證 2,415 組
- 9 筆無效 Qwen 回覆視為 `ABSTAIN`

In [ ]:
device_left, device_right, device_truth = load_benchmark("Device")
device_fuzzy80 = read_pairs("device_fuzzy80.csv")
device_top5 = read_pairs("device_top5_candidates.csv")
device_queue = read_pairs("device_llm_queue.csv")
device_qwen = read_predictions("device_qwen_top5_full3290.csv", device_top5)
assert len(device_top5) == 3290
assert device_queue == device_top5 - device_fuzzy80

device_yes = accepted(device_qwen)
device_fixed = (device_fuzzy80 & device_top5) | (device_yes & device_queue)
device_full = device_fuzzy80 | (device_yes & device_queue)

result_rows += [
    evaluate("Device", "Traditional 80", device_fuzzy80, device_truth, 0),
    evaluate("Device", "Qwen-only Top-5", device_yes, device_truth, len(device_top5)),
    evaluate("Device", "Fixed Hybrid 80", device_fixed, device_truth,
             len(device_queue)),
    evaluate("Device", "Full Hybrid 80", device_full, device_truth,
             len(device_queue)),
]
print("Device — top-5 coverage:", len(device_top5 & device_truth), "/", len(device_truth))
print("Fuzzy outside top-5:", len(device_fuzzy80 - device_top5))
print("Invalid Qwen outputs:", (device_qwen["prediction_eval"] == "ABSTAIN").sum())
display(pd.DataFrame(result_rows[-4:]).drop(columns="Dataset"))

## 4. 三個 Dataset 的 Accuracy / Efficiency 比較

以下表格**由原始 CSV 重新計算**（不是貼上硬編碼的結論）。
Fixed Hybrid 與 Qwen-only Top-5 以相同的候選集合為比較基礎。
Full Hybrid 額外接受 Fuzzy Top-5 以外的配對，須留意不同候選空間帶來的影響。

In [ ]:
summary = pd.DataFrame(result_rows)
assert len(summary) == 12

# 用之前確認過的指標作為 sanity check，防止不小心讀到錯誤檔案。
expected = {
    ("BasketballTeam", "Traditional 80"): .4858,
    ("BasketballTeam", "Qwen-only Top-5"): .5633,
    ("BasketballTeam", "Fixed Hybrid 80"): .5910,
    ("BasketballTeam", "Full Hybrid 80"): .5924,
    ("Country", "Traditional 80"): .2446,
    ("Country", "Qwen-only Top-5"): .4242,
    ("Country", "Fixed Hybrid 80"): .4152,
    ("Country", "Full Hybrid 80"): .3028,
    ("Device", "Traditional 80"): .2955,
    ("Device", "Qwen-only Top-5"): .3604,
    ("Device", "Fixed Hybrid 80"): .3575,
    ("Device", "Full Hybrid 80"): .2775,
}
for (dataset, method), value in expected.items():
    actual = summary.loc[
        summary["Dataset"].eq(dataset) & summary["Method"].eq(method), "F1"
    ].iloc[0]
    assert abs(actual - value) <= .00015, (dataset, method, actual, value)

print("=== ALL THREE DATASETS — FINAL EVALUATION ===")
display(summary)
print("\nF1 comparison")
display(summary.pivot(index="Dataset", columns="Method", values="F1"))

# 存新檔，不覆蓋原本已完成的 summary
new_summary_file = SAVE_DIR / "cross_dataset_summary_clean.csv"
summary.to_csv(new_summary_file, index=False)
print("Saved:", new_summary_file)

In [ ]:
# 僅統計「逐筆已記錄的 Qwen 推論時間」，不是完整 pipeline wall-clock。
def inference_runtime(dataset, qwen_df, queue):
    assert "runtime" in qwen_df.columns, dataset
    pairs = list(zip(qwen_df["left_id"].astype(int), qwen_df["right_id"].astype(int)))
    assert queue <= set(pairs), dataset
    timing = pd.to_numeric(qwen_df["runtime"], errors="raise")
    selected = pd.Series(pairs, index=qwen_df.index).isin(queue)
    qwen_time = timing.sum()
    hybrid_time = timing.loc[selected].sum()
    return {
        "Dataset": dataset,
        "Qwen Calls": len(qwen_df), "Hybrid Calls": len(queue),
        "Qwen Time (s)": round(qwen_time, 2),
        "Hybrid Qwen Time (s)": round(hybrid_time, 2),
        "Inference Time Reduction (%)": round(100 * (1 - hybrid_time / qwen_time), 2),
        "LLM Call Reduction (%)": round(100 * (1 - len(queue) / len(qwen_df)), 2),
    }

runtime_summary = pd.DataFrame([
    inference_runtime("BasketballTeam", basket_qwen, basket_queue),
    inference_runtime("Country", country_qwen, country_queue),
    inference_runtime("Device", device_qwen, device_queue),
])
print("=== INFERENCE TIME (RECORDED PER PAIR) ===")
display(runtime_summary)

new_runtime_file = SAVE_DIR / "runtime_comparison_clean.csv"
runtime_summary.to_csv(new_runtime_file, index=False)
print("Saved:", new_runtime_file)

## 5. Error Analysis（以 Device 為例）

- **False Positives**：語意相近、同品牌或同類別，卻不是相同實體。例如 `PlayStation` vs. `Xbox`。
- **False Negatives**：正確配對可能未被 Embedding Top-5 找到，也可能在 Qwen 驗證時被拒絕。
- **Ground Truth 註解**：少數配對可能涉及實體界定上的歧義；這裡依 benchmark 標籤評分。

In [ ]:
device_fp = device_yes - device_truth
device_retrieval_misses = device_truth - device_top5
device_verification_misses = (device_truth & device_top5) - device_yes

print("=== DEVICE ERROR BREAKDOWN ===")
print("Qwen false positives:", len(device_fp))
print("Ground truth missed by retrieval:", len(device_retrieval_misses))
print("GT retrieved but rejected by Qwen:", len(device_verification_misses))

left_names = device_left.set_index("id")["title"].to_dict()
right_names = device_right.set_index("id")["title"].to_dict()

def describe_pairs(pairs, limit=15):
    return pd.DataFrame([
        {"Left": left_names[lid], "Right": right_names[rid]}
        for lid, rid in sorted(pairs)[:limit]
    ])

print("\nExample false positives")
display(describe_pairs(device_fp, 15))
print("\nAll verification misses")
display(describe_pairs(device_verification_misses, 20))

print("\nFuzzy matches outside Top-5 (additional risk):")
for name, fuzzy_set, candidates, truth in [
    ("Country", country_fuzzy80, country_top5, country_truth),
    ("Device", device_fuzzy80, device_top5, device_truth),
]:
    outside = fuzzy_set - candidates
    print(f"{name}: {len(outside)} pairs; "
          f"GT {len(outside & truth)}, FP {len(outside - truth)}")

## 6. Appendix — 方法實作（可選，不會自動呼叫 Qwen）

### A. Embedding Top-5

下面只是定義方法，與原本三個 Dataset 的實驗一致。**不會重新生成 embeddings**，除非你主動呼叫函式。

In [ ]:
def generate_embedding_top5(left_df, right_df, k=5):
    """Right→Left top-k，使用 cosine similarity 與 normalized embeddings。"""
    import numpy as np
    import torch
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2",
        device="cuda" if torch.cuda.is_available() else "cpu"
    )
    left_emb = embedder.encode(
        left_df["title"].astype(str).tolist(), normalize_embeddings=True,
        batch_size=64, show_progress_bar=True
    )
    right_emb = embedder.encode(
        right_df["title"].astype(str).tolist(), normalize_embeddings=True,
        batch_size=64, show_progress_bar=True
    )
    scores = right_emb @ left_emb.T
    top_indices = np.argsort(-scores, axis=1)[:, :k]
    left_ids = left_df["id"].astype(int).to_numpy()
    right_ids = right_df["id"].astype(int).to_numpy()
    return {
        (int(left_ids[j]), int(right_ids[i]))
        for i, indices in enumerate(top_indices)
        for j in indices
    }

print("Embedding function defined (not executed).")

### B. Qwen generic prompt + verification

這裡保留原本 `Big_data (2)` 的**一份** Qwen 實作，方便檢查研究方法。
只有手動呼叫 `load_qwen_model()` 才會載入 GPU 模型；只有手動呼叫 `verify_pair(...)` 才會推論。

**環境註記：** 原本完成實驗的 Colab 曾使用 `transformers==4.57.1`、`tokenizers==0.22.1`。
請勿在已經能正常運行的 Colab 直接執行 `pip install -U transformers` 或中途解除安裝 Transformers。

In [ ]:
SYSTEM_PROMPT = """You are an entity matching assistant.
Determine whether two names refer to the same real-world entity.
Consider:
- Abbreviations and aliases
- Historical renaming
- Relocation
- Spelling and formatting differences
Similar names do not necessarily refer to the same entity.
If there is insufficient evidence, answer UNSURE.
Answer with exactly one word: YES, NO, or UNSURE."""

def load_qwen_model():
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    if not torch.cuda.is_available():
        raise RuntimeError("請先在 Colab 啟用 GPU；目前不需要重跑 Qwen。")
    model_name = "Qwen/Qwen2.5-1.5B-Instruct"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, torch_dtype=torch.float16
    ).to("cuda")
    model.eval()
    return tokenizer, model

def verify_pair(name_a, name_b, tokenizer, model):
    import re
    import torch
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Entity A: {name_a}\nEntity B: {name_b}"},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=12, do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    raw = tokenizer.decode(
        output[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()
    match = re.match(r"^(YES|NO|UNSURE)\b", raw.upper())
    return (match.group(1) if match else "INVALID"), raw

print("Qwen functions defined (model not loaded; no inference).")

---
## Final Findings & Limitations

1. **Efficiency trade-off**：Fixed Hybrid 在三個 Dataset 皆減少 LLM calls（BasketballTeam **9.04%**、Country **14.09%**、Device **26.60%**）。
2. **Accuracy is not guaranteed**：Fixed Hybrid 在 BasketballTeam F1 較好，但在 Country / Device 略低於 Qwen-only。
3. **Beware of fuzzy false positives**：Full Hybrid 直接接受 Top-5 以外的 Fuzzy 配對，會大幅降低 Country / Device 的 Precision。
4. **Retrieval recall ceiling**：Device 的 Top-5 缺失 **113** 組 GT；Qwen 驗證另外漏掉 **5** 組。
5. **Runtime limitation**：目前的推論時間是每筆 Qwen 記錄加總，**不是 end-to-end runtime**。完整比較仍須納入資料讀取、Fuzzy、Embedding 與模型載入成本。
6. **Reproducibility**：正式結果來自既有 Qwen CSV；Appendix 提供原演算法的獨立函式，但不會在本次重跑。

**請保留舊 Notebook 作為開發歷史，不要把它的除錯 Cell 再貼回正式版。**